# 앞선 작업의 문제점
- docs 문서 중 이미지 표를 docx2txtLoader가 정상적으로 읽지 못하여 결과가 이상하게 나옴
- 이미지 표를 table형태로 바꾼후 index 재구성

In [1]:
%pip install docx2txt langchain-community --quiet


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain-text-splitters --quiet


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install langchain-upstage python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./data/tax_with_table.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

## pinecone 활용한 벡터 DB 저장

In [8]:
%pip install --upgrade --quiet langchain-pinecone \
     langchain-openai langchain 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = "tax-table-index"
pinecone_api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

/Users/psh/.pyenv/versions/llm-application/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* 인덱스 생성 필요 - 현재 과제에서는 pinecone console에서 진행함

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [11]:
from langchain_core.prompts import ChatPromptTemplate

query = "연봉 5000만원 직장이 기준 소득세는 얼마인가요?"

prompt = ChatPromptTemplate.from_messages([
    ('system', """소득세법에 대한 질문에 아래 context를 근거로 답변하세요.
질문에 대한 직접적인 문장이 없더라도, 공제 규정과 세율표를 활용해 단계적으로 계산할 수 있다면 계산 과정을 보여주고 답하세요.
context에서 전혀 근거를 찾을 수 없을 때만 모른다고 답하세요.

    [Context]
    {context}
    """),
    ('human', '{question}')
])

In [12]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain.invoke({"query": query})

In [13]:
ai_message['result']

'연봉 5,000만 원을 기준으로 소득세를 계산하기 위해서는 종합소득 과세표준에 맞춰 세율을 적용해야 하고, 가능한 공제를 고려해야 합니다.\n\n소득세 계산은 다음 단계로 이루어집니다.\n\n1. **과세표준에 해당하는 세율 적용**:\n   - 연봉 5,000만 원은 1,400만 원을 초과하고 5,000만 원 이하에 해당합니다. 따라서 해당 구간의 세율을 적용하면 됩니다.\n   - 세율: 84만 원 + (1,400만 원을 초과하는 금액의 15%)\n   - 1,400만 원 초과 금액 = 5,000만 원 - 1,400만 원 = 3,600만 원\n   - 3,600만 원의 15% = 540만 원\n   - 따라서, 소득세 = 84만 원 + 540만 원 = 624만 원\n\n2. **근로소득세액공제 적용**:\n   - 총급여 5,000만 원은 3,300만 원 초과 7,000만 원 이하 구간에 해당합니다.\n   - 공제액 = 74만 원 - [(총급여 - 3,300만 원) × 8/1000] \n   - 공제액 = 74만 원 - [(5,000만 원 - 3,300만 원) × 8/1000]\n   - 공제액 = 74만 원 - (1,700만 원 × 0.008) = 74만 원 - 13.6만 원 = 60.4만 원 (66만 원보다 적으므로 66만 원 적용)\n\n3. **소득세의 최종 금액**:\n   - 소득세 = 624만 원 - 66만 원 = 558만 원\n\n따라서 연봉 5,000만 원인 직장의 기준 소득세는 558만 원입니다.'